In [12]:
pip install torch torch-geometric -q

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import QM9
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv, global_mean_pool

In [14]:
import importlib
import torch_geometric.datasets.qm9 as qm9_module
import torch
importlib.reload(qm9_module)
dist_mean,dist_std=0,0
def add_distances(data):
  global dist_mean,dist_std
  row, col = data.edge_index
  dist = torch.norm(data.pos[row] - data.pos[col], dim=-1, keepdim=True)
  dist_mean = dist.mean()
  dist_std  = dist.std()
  dist_std  = dist_std if dist_std > 0 else torch.tensor(1.0)
  dist_normed  = (dist - dist_mean) / dist_std
  data.edge_attr = torch.cat([data.edge_attr, dist_normed], dim=-1)  # [E, 5]
  return data
from torch_geometric.datasets import QM9
dataset = QM9(root='data/QM9',transform=add_distances)
print(f"✅ Loaded {len(dataset)} molecules")

✅ Loaded 130831 molecules


In [15]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dataset    = QM9(root='data/QM9',transform=add_distances).shuffle()
train_data = dataset[:104000]
test_data  = dataset[104000:]

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=32)

In [16]:
train_y = torch.cat([data.y[:, :16] for data in train_data], dim=0)
target_mean = train_y.mean(dim=0).to(device)
target_std  = train_y.std(dim=0).to(device)
target_std[target_std == 0] = 1.0

In [17]:
class GINENet(nn.Module):
  def __init__(self,node_dim=11, edge_dim=5, hidden = 64, out_dim=16):
    super().__init__()
    self.node_emb=nn.Linear(node_dim,hidden)
    def make_mlp(in_dim, out_dim):
      return nn.Sequential(
                nn.Linear(in_dim, out_dim),
                nn.ReLU(),
                nn.Linear(out_dim, out_dim)
            )

    self.conv1 = GINEConv(make_mlp(hidden, hidden), edge_dim=edge_dim)
    self.conv2 = GINEConv(make_mlp(hidden, hidden), edge_dim=edge_dim)
    self.conv3 = GINEConv(make_mlp(hidden, hidden), edge_dim=edge_dim)

    self.lin1 = nn.Linear(hidden, 32)
    self.lin2 = nn.Linear(32, out_dim)
  def forward(self,data):
    x= data.x
    edge_index = data.edge_index
    edge_attr  = data.edge_attr
    batch      = data.batch
    x = F.relu(self.node_emb(x))
    x = F.relu(self.conv1(x, edge_index, edge_attr))
    x = F.relu(self.conv2(x, edge_index, edge_attr))
    x = F.relu(self.conv3(x, edge_index, edge_attr))
    x = global_mean_pool(x, batch)

    x = F.relu(self.lin1(x))
    x = self.lin2(x)
    return x



In [18]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = GINENet().to(device)
optim  = torch.optim.Adam(model.parameters(), lr=4e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optim,
    mode='min',      # lower Test MAE is better
    factor=0.5,      # halve the learning rate
    patience=5,      # wait 5 epochs before reducing
    min_lr=1e-5      # never go below this
)

In [19]:
def train(loader):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optim.zero_grad()
        pred  = model(batch)
        true = (batch.y[:, :16] - target_mean) / target_std  # normalise targets

        loss  = F.mse_loss(pred, true)
        loss.backward()
        optim.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(loader.dataset)

@torch.no_grad()
def test(loader):
    model.eval()
    total_mae = 0
    for batch in loader:
        batch    = batch.to(device)
        pred = model(batch) * target_std + target_mean   # denormalise back to real units

        true     = batch.y[:, :16]
        total_mae += F.l1_loss(pred, true, reduction='sum').item()
    return total_mae / len(loader.dataset)


@torch.no_grad()
def evaluate(loader):
    model.eval()

    all_preds = []
    all_true  = []

    for batch in loader:
        batch = batch.to(device)
        pred  = model(batch) * target_std + target_mean   # denormalise
        true  = batch.y[:, :16]

        all_preds.append(pred.cpu())
        all_true.append(true.cpu())

    all_preds = torch.cat(all_preds, dim=0)   # [N, 19]
    all_true  = torch.cat(all_true,  dim=0)   # [N, 19]

    # MAE per property
    mae  = (all_preds - all_true).abs().mean(dim=0)         # [19]

    # RMSE per property
    rmse = ((all_preds - all_true) ** 2).mean(dim=0).sqrt() # [19]

    # R² per property
    ss_res = ((all_true - all_preds) ** 2).sum(dim=0)
    ss_tot = ((all_true - all_true.mean(dim=0)) ** 2).sum(dim=0)
    r2     = 1 - ss_res / ss_tot                            # [19]

    return mae, rmse, r2

In [20]:
print(f"{'Epoch':>5}  {'Train MSE':>10}  {'Test MAE':>10}")
print("-" * 32)
for epoch in range(1, 51):
    train_loss = train(train_loader)
    test_mae   = test(test_loader)
    scheduler.step(test_mae)
    if epoch % 5 == 0:
      current_lr = optim.param_groups[0]['lr']
      print(f"{epoch:>5}  {train_loss:>10.4f}  {test_mae:>10.4f}  {current_lr:>10.6f}")


Epoch   Train MSE    Test MAE
--------------------------------
    5      0.2759   1668.7461    0.000400
   10      0.2182   1636.8069    0.000400
   15      0.1867   1466.6654    0.000400
   20      0.1699   1348.8828    0.000400
   25      0.1510   1375.6753    0.000200
   30      0.1439   1330.5240    0.000200
   35      0.1346   1225.9437    0.000100
   40      0.1311   1170.7870    0.000100
   45      0.1278   1217.3570    0.000100
   50      0.1225   1176.3675    0.000050


In [22]:
mae, rmse, r2 = evaluate(test_loader)

property_names = [
    'Dipole moment',
    'Isotropic polarizability',
    'HOMO energy',
    'LUMO energy',
    'HOMO-LUMO gap',
    'Electronic spatial extent',
    'ZPVE',
    'U0',
    'U',
    'H',
    'G',
    'Cv',
    'U0_atom',
    'U_atom',
    'H_atom',
    'G_atom'
]

print(f"{'Property':<30}  {'MAE':>10}  {'RMSE':>10}  {'R²':>10}")
print("-" * 65)
for i, name in enumerate(property_names):
    print(f"{name:<30}  {mae[i]:>10.4f}  {rmse[i]:>10.4f}  {r2[i]:>10.4f}")

Property                               MAE        RMSE          R²
-----------------------------------------------------------------
Dipole moment                       0.6153      0.8319      0.6949
Isotropic polarizability            2.0945      3.1123      0.8542
HOMO energy                         0.1766      0.2300      0.8524
LUMO energy                         0.1985      0.2615      0.9576
HOMO-LUMO gap                       0.2627      0.3457      0.9270
Electronic spatial extent          79.4683    111.5044      0.8429
ZPVE                                0.1267      0.1878      0.9559
U0                                271.1215    421.1848      0.8484
U                                 271.0718    421.1972      0.8484
H                                 271.2337    421.1394      0.8485
G                                 271.1021    421.2212      0.8484
Cv                                  0.9801      1.4103      0.8780
U0_atom                             2.0050      3.1392      0.9